### 데이터 정리 자동화

- 제조 센터 데이터의 컬럼명 표준화
- 문자열 형태의 시간 데이터를 datetime 형식으로 변환
- 압력, 속도 같은 센서 단위를 변환
- 결측치와 이상치를 처리 
- 전체 데이터 정리 과정을 함수로 자동화 

In [1]:
# 라이브러리 및 파일 불러오기

import numpy as np 
import pandas as pd 
from datetime import datetime, timedelta

file_path = r"C:\Users\user\Desktop\python_campus\dev\data\raw\week07_day04_raw_messy_sensor.csv"

sensor_df = pd.read_csv(file_path, encoding = "utf-8-sig")

sensor_df.head()

,Time Stamp,Machine ID,TEMP_C,Pressure[bar],speed(m/s),Humidity %,VIB(mm/s),CURRENT_A,status
0,2026/07/01 09:00:00,MC_03,28.36,3.48,5.28,51.35,3.89,10.39,NaN
1,2026/07/01 09:01:00,MC_01,24.20,2.82,4.64,67.22,3.62,6.34,ng
2,2026/07/01 09:02:00,MC_03,27.67,2.45,4.11,64.43,2.42,6.92,OK
3,2026/07/01 09:03:00,MC_03,26.59,2.79,4.39,59.31,4.95,7.76,ng
4,2026/07/01 09:04:00,MC_01,23.92,2.56,4.67,61.00,1.58,7.82,ok


In [2]:
sensor_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 121 entries, 0 to 120
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0    Time Stamp    121 non-null    str    
 1   Machine ID     121 non-null    str    
 2   TEMP_C         120 non-null    float64
 3   Pressure[bar]  120 non-null    float64
 4   speed(m/s)     120 non-null    float64
 5   Humidity %     121 non-null    float64
 6   VIB(mm/s)      121 non-null    float64
 7   CURRENT_A      121 non-null    float64
 8   status         112 non-null    str    
dtypes: float64(6), str(3)
memory usage: 8.6 KB


In [3]:
sensor_df.describe()

,TEMP_C,Pressure[bar],speed(m/s),Humidity %,VIB(mm/s),CURRENT_A
count,120.000000,120.000000,120.000000,121.000000,121.000000,121.000000
mean,34.350000,2.801667,5.025167,60.083967,3.836694,8.806364
std,88.820858,0.591502,0.745815,9.788240,8.760971,1.648388
min,20.690000,-3.000000,-1.000000,46.300000,1.140000,-5.000000
25%,24.990000,2.677500,4.780000,56.380000,2.530000,8.120000
50%,26.320000,2.820000,5.090000,59.040000,3.080000,8.950000
75%,27.705000,3.022500,5.447500,62.900000,3.640000,9.620000
max,999.000000,3.480000,6.140000,150.000000,99.000000,11.160000


In [4]:
# 컬럼별 결측치 개수를 확인 

missing_count = sensor_df.isna().sum()  # isna : N/A 비어있는 셀 확인, sum : 합계 계산 

missing_count

 Time Stamp      0
Machine ID       0
TEMP_C           1
Pressure[bar]    1
speed(m/s)       1
Humidity %       0
VIB(mm/s)        0
CURRENT_A        0
status           9
dtype: int64

In [5]:
sensor_df.columns

Index([' Time Stamp ', 'Machine ID', 'TEMP_C', 'Pressure[bar]', 'speed(m/s)',
       'Humidity %', 'VIB(mm/s)', 'CURRENT_A', 'status '],
      dtype='str')

In [6]:
# 컬럼명 표준화 하기

# 소문자 통일
# 공백 대신 _ 사용 
# 단위는 컬럼명에 명확히 표시 
# 의미가 분명한 이름으로 표시 

# 컬럼명 표준 이름으로 바꾸기 위해 딕셔너리 사용 (생성 후 매핑 진행)

column_map = {
    " Time Stamp " : "timestamp",
    "Machine ID" : "machine_id", 
    "TEMP_C" : "temperature_c", 
    "Pressure[bar]" : "pressure_bar",
    "speed(m/s)" : "speed_mps",
    "Humidity %" : "humidity_pct",
    "VIB(mm/s)" : "vibration_mm_s",
    "CURRENT_A" : "current_a",
    "status " : "status",
    }

standard_df = sensor_df.rename(columns = column_map)

standard_df.columns

Index(['timestamp', 'machine_id', 'temperature_c', 'pressure_bar', 'speed_mps',
       'humidity_pct', 'vibration_mm_s', 'current_a', 'status'],
      dtype='str')

In [7]:
# 시간 데이터 변환하기 (문자열 timestamp -> datetime 으로 변환)

standard_df["timestamp"] = pd.to_datetime(
    standard_df["timestamp"],
    errors = "coerce" # 변환할 수 없는 값 NaT로 변환 
)

standard_df["timestamp"].dtype

dtype('<M8[us]')

In [8]:
# 문자열을 datetime으로 변환 시, 이상한 값이 NaT로 바뀌었는지 확인 

standard_df["timestamp"].isna().sum()

np.int64(1)

In [9]:
bad_time_df = standard_df[standard_df["timestamp"].isna()] # 해당 값만 불러오겠다는 의미 (빈 값)
bad_time_df

,timestamp,machine_id,temperature_c,pressure_bar,speed_mps,humidity_pct,vibration_mm_s,current_a,status
50,NaT,MC_02,26.2,2.58,5.25,60.14,2.72,11.16,OK


In [10]:
# 단위 변환하기 

# 압력 bar -> kPa (1bar -> 100kPa)
# 속도 m/s -> m/min (1m/s -> 60m/min)

standard_df["pressure_kpa"] = standard_df["pressure_bar"] * 100
standard_df["speed_mpm"] = standard_df["speed_mps"] * 60

standard_df[["pressure_bar", "pressure_kpa", "speed_mps", "speed_mpm"]].head()

,pressure_bar,pressure_kpa,speed_mps,speed_mpm
0,3.48,348.0,5.28,316.8
1,2.82,282.0,4.64,278.4
2,2.45,245.0,4.11,246.6
3,2.79,279.0,4.39,263.4
4,2.56,256.0,4.67,280.2


In [14]:
# 문자 데이터 정리하기 

# status 컬럼에 OK, NG, ok, ng 처럼 공백과 대소문자가 섞여 있어 처리 필요 

standard_df["status"] = standard_df["status"].fillna("UNKNOWN") # 빈자리를 unknown으로 채움 
standard_df["status"] = standard_df["status"].astype(str).str.strip().str.upper() # 공백 없애기, 대문자 변환

# OK, NG가 아닌 값은 UNKNOWN으로 통일 
standard_df.loc[~standard_df["status"].isin(["OK", "NG"]), "status"] = "UNKNOWN"

standard_df["status"].value_counts(dropna = False)

status
OK         78
NG         34
UNKNOWN     9
Name: count, dtype: int64

In [18]:
# 중복 데이터 제거하기

duplicate_count = standard_df.duplicated().sum()
duplicate_count

np.int64(1)

In [21]:
standard_df = standard_df.drop_duplicates() # 중복행 제거
standard_df = standard_df.reset_index(drop=True) # 드롭된 행을 인덱스 재정렬 

standard_df.shape

(120, 11)

In [22]:
# 결측치 처리하기 

# 시간 결측치 : 분석이 어려우므로 행 제거 
# 숫자 결측치 : 중앙값으로 채우기 #### 실제 존재하는 값인 중앙값으로 채움 
# 문자 결측치 : UNKNOWN으로 채우기 

standard_df.isna().sum()

timestamp         1
machine_id        0
temperature_c     1
pressure_bar      1
speed_mps         1
humidity_pct      0
vibration_mm_s    0
current_a         0
status            0
pressure_kpa      1
speed_mpm         1
dtype: int64

In [28]:
# timestamp 결측행 제거

before_count = len(standard_df)
standard_df = standard_df.dropna(subset=["timestamp"])
after_count = len(standard_df)

print("timestamp 결측으로 제거된 행 수 :", before_count - after_count)

timestamp 결측으로 제거된 행 수 : 1


In [29]:
# 숫자형 컬럼의 결측치를 중앙값으로 채우기

numeric_cols = ["temperature_c",
                "pressure_bar", 
                "speed_mps",
                "humidity_pct",
                "vibration_mm_s",
                "current_a",
                "pressure_kpa",
                "speed_mpm"]

for col in numeric_cols:
    median_value = standard_df[col].median() # 각 컬럼의 중앙값
    standard_df[col] = standard_df[col].fillna(median_value) # 중앙값으로 채움 
    print(f"{col} 결측치 -> 중앙값 {median_value:.2f}로 대체")

standard_df.isna().sum()

temperature_c 결측치 -> 중앙값 26.32로 대체
pressure_bar 결측치 -> 중앙값 2.82로 대체
speed_mps 결측치 -> 중앙값 5.08로 대체
humidity_pct 결측치 -> 중앙값 59.04로 대체
vibration_mm_s 결측치 -> 중앙값 3.08로 대체
current_a 결측치 -> 중앙값 8.94로 대체
pressure_kpa 결측치 -> 중앙값 282.00로 대체
speed_mpm 결측치 -> 중앙값 304.80로 대체


timestamp         0
machine_id        0
temperature_c     0
pressure_bar      0
speed_mps         0
humidity_pct      0
vibration_mm_s    0
current_a         0
status            0
pressure_kpa      0
speed_mpm         0
dtype: int64

In [31]:
# 이상치 처리하기 (정상 범위를 벗어난 행을 제거)

# 센서별 정상 범위를 딕셔너리 정의 

valid_ranges = {
    "temperature_c" :(0,100),
    "pressure_bar" : (0,10),
    "speed_mps" : (0,20),
    "humidity_pct" : (0,100),
    "vibration_mm_s" : (0,20),
    "current_a" : (0,30)
}

outlier_summary = []

for col, (min_value, max_value) in valid_ranges.items():
    outlier_count = ((standard_df[col] < min_value) | (standard_df[col] > max_value)).sum()
    # 최소값과 최대값을 벗어나는 값에 대한 집계
    outlier_summary.append({
        "column" : col,
        "min_value" : min_value,
        "max_value" : max_value,
        "outlier_count" : int(outlier_count)
    })

print(outlier_summary)
outlier_report = pd.DataFrame(outlier_summary)

[{'column': 'temperature_c', 'min_value': 0, 'max_value': 100, 'outlier_count': 1}, {'column': 'pressure_bar', 'min_value': 0, 'max_value': 10, 'outlier_count': 1}, {'column': 'speed_mps', 'min_value': 0, 'max_value': 20, 'outlier_count': 1}, {'column': 'humidity_pct', 'min_value': 0, 'max_value': 100, 'outlier_count': 1}, {'column': 'vibration_mm_s', 'min_value': 0, 'max_value': 20, 'outlier_count': 1}, {'column': 'current_a', 'min_value': 0, 'max_value': 30, 'outlier_count': 1}]


In [32]:
outlier_report

,column,min_value,max_value,outlier_count
0,temperature_c,0,100,1
1,pressure_bar,0,10,1
2,speed_mps,0,20,1
3,humidity_pct,0,100,1
4,vibration_mm_s,0,20,1
5,current_a,0,30,1


In [33]:
# 정상 범위를 모두 만족하는 행만 남기기 

clean_df = standard_df.copy()

for col, (min_value, max_value) in valid_ranges.items() :
    clean_df = clean_df[
        (clean_df[col] >= min_value) & (clean_df[col] <= max_value)
    ]

clean_df = clean_df.reset_index(drop=True)

print("이상치 제거 전 데이터 크기 : ", standard_df.shape)
print("이상치 제거 후 데이터 크기 : ", clean_df.shape)

이상치 제거 전 데이터 크기 :  (119, 11)
이상치 제거 후 데이터 크기 :  (113, 11)


In [34]:
# 이상치 리포트를 파일로 저장하기

outlier_report_file = r"C:\Users\user\Desktop\python_campus\dev\data\result\outlier_report.csv"

outlier_report.to_csv(outlier_report_file, index=False, encoding="utf-8-sig")

In [35]:
pd.read_csv(outlier_report_file, encoding="utf-8-sig")

,column,min_value,max_value,outlier_count
0,temperature_c,0,100,1
1,pressure_bar,0,10,1
2,speed_mps,0,20,1
3,humidity_pct,0,100,1
4,vibration_mm_s,0,20,1
5,current_a,0,30,1


In [36]:
# 품질 판정 컬럼 만들기 

#전처리된 데이터를 이용해 간단한 Rule 기반 품질 판정을 만들어 봅니다.

# NG 조건 설정
# 온도 temperature_c > 30 
# 압력 pressure_bar > 3.2
# 속도 speed_mps < 4.0
# 습도 humidity_pct > 70
# 진동 vibration_mm_s > 6.0
# 전류 current_a > 12.0

# 하나라도 만족하면 NG, 모두 만족하지 않으면 OK 판정

In [39]:
# 품질 판정 컬럼 만들기

clean_df["quality"] = "OK"

clean_df.loc[
    (clean_df["temperature_c"] > 30) |
    (clean_df["pressure_bar"] > 3.2) |
    (clean_df["speed_mps"] < 4.0) |
    (clean_df["humidity_pct"] > 70) |
    (clean_df["vibration_mm_s"] > 6.0) |
    (clean_df["current_a"] > 12.0),
    "quality"
] = "NG"

clean_df["quality"].value_counts()

quality
OK    93
NG    20
Name: count, dtype: int64

In [41]:
# 품질 리포트 만들기 

final_total_count = len(clean_df)
final_ok_count = (clean_df["quality"]=="OK").sum()
final_ng_count = (clean_df["quality"]=="NG").sum()
final_ng_ratio = final_ng_count / final_total_count * 100

quality_report = pd.DataFrame({
    "total_count" : [final_total_count],
    "ok_count" : [final_ok_count],
    "ng_count" : [final_ng_count],
    "ng_ratio_pct" : [round(final_ng_ratio, 2)]
})

quality_report

,total_count,ok_count,ng_count,ng_ratio_pct
0,113,93,20,17.7


In [44]:
# 설비별 품질 요약 리포트 만들기 groupby

machine_report = clean_df.groupby("machine_id").agg(
    total_count = ("quality", "count"),
    avg_temperature = ("temperature_c", "mean"),
    avg_pressure = ("pressure_bar", "mean"),
    ng_count = ("quality", lambda x : (x=="NG").sum())
).reset_index()

machine_report["ng_ratio_pct"] = (machine_report["ng_count"] / machine_report["total_count"]*100).round(2)

machine_report

,machine_id,total_count,avg_temperature,avg_pressure,ng_count,ng_ratio_pct
0,MC_01,36,25.733611,2.834444,8,22.22
1,MC_02,37,26.237838,2.844324,3,8.11
2,MC_03,40,26.567750,2.874000,9,22.50


In [45]:
machine_report_file = r"C:\Users\user\Desktop\python_campus\dev\data\result\machine_report.csv"

machine_report.to_csv(machine_report_file, index=False, encoding="utf-8-sig")

In [46]:
pd.read_csv(machine_report_file, encoding="utf-8-sig")

,machine_id,total_count,avg_temperature,avg_pressure,ng_count,ng_ratio_pct
0,MC_01,36,25.733611,2.834444,8,22.22
1,MC_02,37,26.237838,2.844324,3,8.11
2,MC_03,40,26.567750,2.874000,9,22.50


In [48]:
# 전체 처리 함수화하기

# 파일을 읽어 오는 함수 만들기 

def load_senor_data(file_path) :
    """ csv 파일을 읽어 데이터 프레임으로 반환함 """

    df = pd.read_csv(file_path, encoding="utf-8-sig")
    return df 


In [49]:
file_path = r"C:\Users\user\Desktop\python_campus\dev\data\raw\week07_day04_raw_messy_sensor.csv"

df = load_senor_data(file_path)
df.head()

,Time Stamp,Machine ID,TEMP_C,Pressure[bar],speed(m/s),Humidity %,VIB(mm/s),CURRENT_A,status
0,2026/07/01 09:00:00,MC_03,28.36,3.48,5.28,51.35,3.89,10.39,NaN
1,2026/07/01 09:01:00,MC_01,24.20,2.82,4.64,67.22,3.62,6.34,ng
2,2026/07/01 09:02:00,MC_03,27.67,2.45,4.11,64.43,2.42,6.92,OK
3,2026/07/01 09:03:00,MC_03,26.59,2.79,4.39,59.31,4.95,7.76,ng
4,2026/07/01 09:04:00,MC_01,23.92,2.56,4.67,61.00,1.58,7.82,ok


In [50]:
# 컬럼명 표준화 함수 만들기 

def standardize_columns(df) :
    """ 제조 센터 데이터의 컬럼명을 표준화함 """
    
    column_map = {
    " Time Stamp " : "timestamp",
    "Machine ID" : "machine_id", 
    "TEMP_C" : "temperature_c", 
    "Pressure[bar]" : "pressure_bar",
    "speed(m/s)" : "speed_mps",
    "Humidity %" : "humidity_pct",
    "VIB(mm/s)" : "vibration_mm_s",
    "CURRENT_A" : "current_a",
    "status " : "status",
    }
    df = df.rename(columns=column_map)
    return df 

In [51]:
standard_df = standardize_columns(df)
standard_df.head()

,timestamp,machine_id,temperature_c,pressure_bar,speed_mps,humidity_pct,vibration_mm_s,current_a,status
0,2026/07/01 09:00:00,MC_03,28.36,3.48,5.28,51.35,3.89,10.39,NaN
1,2026/07/01 09:01:00,MC_01,24.20,2.82,4.64,67.22,3.62,6.34,ng
2,2026/07/01 09:02:00,MC_03,27.67,2.45,4.11,64.43,2.42,6.92,OK
3,2026/07/01 09:03:00,MC_03,26.59,2.79,4.39,59.31,4.95,7.76,ng
4,2026/07/01 09:04:00,MC_01,23.92,2.56,4.67,61.00,1.58,7.82,ok


In [52]:
# 시간 변환 함수 만들기 

def convert_time_and_units(df) :
    """timestamp 변환, 압력/속도 단위 변환을 수행함"""

    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    df["pressure_kpa"] = df["pressure_bar"] * 100
    df["speed_mpm"] = df["speed_mps"] * 60
    return df


In [53]:
standard_df = convert_time_and_units(standard_df)
standard_df.head()

,timestamp,machine_id,temperature_c,pressure_bar,speed_mps,humidity_pct,vibration_mm_s,current_a,status,pressure_kpa,speed_mpm
0,2026-07-01 09:00:00,MC_03,28.36,3.48,5.28,51.35,3.89,10.39,NaN,348.0,316.8
1,2026-07-01 09:01:00,MC_01,24.20,2.82,4.64,67.22,3.62,6.34,ng,282.0,278.4
2,2026-07-01 09:02:00,MC_03,27.67,2.45,4.11,64.43,2.42,6.92,OK,245.0,246.6
3,2026-07-01 09:03:00,MC_03,26.59,2.79,4.39,59.31,4.95,7.76,ng,279.0,263.4
4,2026-07-01 09:04:00,MC_01,23.92,2.56,4.67,61.00,1.58,7.82,ok,256.0,280.2


In [54]:
# status 컬럼의 텍스트 처리 함수 

def clean_text_columns(df) : 
    """status 컬럼의 공백과 대소문자를 정리함"""

    df["status"] = df["status"].fillna("UNKNOWN")
    df["status"] = df["status"].astype(str).str.strip().str.upper()
    df.loc[~df["status"].isin(["OK", "NG"]), "status"] = "UNKNOWN"
    return df 

In [55]:
standard_df = clean_text_columns(standard_df)

standard_df["status"]

0      UNKNOWN
1           NG
2           OK
3           NG
4           OK
        ...   
116         NG
117         OK
118         OK
119         NG
120    UNKNOWN
Name: status, Length: 121, dtype: str

In [56]:
# 결측치 처리하는 함수

def handle_missing_values(df) :
    """ 시간 결측은 제거하고, 숫자 결측은 중앙값으로 대체함 """

    df = df.dropna(subset=["timestamp"])

    numeric_cols = ["temperature_c",
                "pressure_bar", 
                "speed_mps",
                "humidity_pct",
                "vibration_mm_s",
                "current_a",
                "pressure_kpa",
                "speed_mpm"]
    
    for col in numeric_cols :
        df[col] = df[col].fillna(df[col].median())

    return df 

In [57]:
standard_df = handle_missing_values(standard_df)

standard_df.isna().sum()

timestamp         0
machine_id        0
temperature_c     0
pressure_bar      0
speed_mps         0
humidity_pct      0
vibration_mm_s    0
current_a         0
status            0
pressure_kpa      0
speed_mpm         0
dtype: int64

In [58]:
# 이상치 제거하는 함수 만들기 

def remove_outliers(df, valid_ranges) :
    """정상 범위를 벗어난 이상치 행을 제거하고 이상치 리포트를 반환함"""

    outlier_summary = []

    for col, (min_value, max_value) in valid_ranges.items():
        outlier_count = ((df[col] < min_value) | (df[col] > max_value)).sum()
        outlier_summary.append({
            "column" : col,
            "min_value" : min_value, 
            "max_value" : max_value,
            "outlier_count" : int(outlier_count)
        })

        df = df[(df[col]>=min_value) & (df[col] <= max_value)]

    outlier_report = pd.DataFrame(outlier_summary)
    df = df.reset_index(drop=True)

    return df, outlier_report

In [62]:
valid_ranges = {
    "temperature_c" :(0,100),
    "pressure_bar" : (0,10),
    "speed_mps" : (0,20),
    "humidity_pct" : (0,100),
    "vibration_mm_s" : (0,20),
    "current_a" : (0,30)
}

clean_df, outlier_report = remove_outliers(standard_df, valid_ranges)

clean_df.head()

,timestamp,machine_id,temperature_c,pressure_bar,speed_mps,humidity_pct,vibration_mm_s,current_a,status,pressure_kpa,speed_mpm
0,2026-07-01 09:00:00,MC_03,28.36,3.48,5.28,51.35,3.89,10.39,UNKNOWN,348.0,316.8
1,2026-07-01 09:01:00,MC_01,24.20,2.82,4.64,67.22,3.62,6.34,NG,282.0,278.4
2,2026-07-01 09:02:00,MC_03,27.67,2.45,4.11,64.43,2.42,6.92,OK,245.0,246.6
3,2026-07-01 09:03:00,MC_03,26.59,2.79,4.39,59.31,4.95,7.76,NG,279.0,263.4
4,2026-07-01 09:04:00,MC_01,23.92,2.56,4.67,61.00,1.58,7.82,OK,256.0,280.2


In [67]:
clean_df[["temperature_c",
         "pressure_bar",
         "speed_mps",
         "humidity_pct",
         "vibration_mm_s",
         "current_a"]].describe()

,temperature_c,pressure_bar,speed_mps,humidity_pct,vibration_mm_s,current_a
count,114.000000,114.000000,114.000000,114.000000,114.000000,114.000000
mean,26.213246,2.857193,5.084649,59.469561,3.062807,8.862895
std,1.941648,0.256999,0.500953,5.375824,0.816447,1.036202
min,20.690000,2.300000,3.770000,46.300000,1.140000,6.340000
25%,24.870000,2.682500,4.780000,56.677500,2.522500,8.075000
50%,26.320000,2.825000,5.100000,59.165000,3.080000,8.935000
75%,27.650000,3.030000,5.470000,63.177500,3.655000,9.577500
max,32.740000,3.480000,6.140000,73.920000,5.110000,11.080000


In [63]:
outlier_report

,column,min_value,max_value,outlier_count
0,temperature_c,0,100,1
1,pressure_bar,0,10,1
2,speed_mps,0,20,1
3,humidity_pct,0,100,1
4,vibration_mm_s,0,20,1
5,current_a,0,30,1


In [65]:
# 품질 판정과 리포트 생성 함수 만들기 

def add_quality_result(df) :
    """rule 기반 OK,NG 품질을 결정함  """

    df["quality"] = "OK"
    clean_df.loc[
    (clean_df["temperature_c"] > 30) |
    (clean_df["pressure_bar"] > 3.2) |
    (clean_df["speed_mps"] < 4.0) |
    (clean_df["humidity_pct"] > 70) |
    (clean_df["vibration_mm_s"] > 6.0) |
    (clean_df["current_a"] > 12.0),
    "quality"
    ] = "NG"

    return df 

In [66]:
clean_df = add_quality_result(clean_df)
clean_df["quality"].value_counts()

quality
OK    93
NG    21
Name: count, dtype: int64

In [68]:
# NG 비율을 리포트로 만드는 함수 

def make_quality_report(df) :
    """ 전체 OK,NG 개수와 NG 비율 리포트를 만듦 """

    total_count = len(df)
    ok_count = (df["quality"]=="OK").sum()
    ng_count = (df["quality"]=="NG").sum()
    ng_ratio = ng_count / total_count * 100

    report = pd.DataFrame({
        "total_count" : [total_count],
        "ok_count" : [ok_count],
        "ng_count" : [ng_count],
        "ng_ratio" : [round(ng_ratio, 2)]
    })

    return report 

In [69]:
result_report = make_quality_report(clean_df)

result_report

,total_count,ok_count,ng_count,ng_ratio
0,114,93,21,18.42


In [70]:
# 지금까지 만든 함수를 하나로 연결하는 전처리 자동화 함수 만들기 

def run_processing(input_file) :
    """ 제조 센서 데이터 정리 전체 과정을 실행함 """

    valid_ranges = {
    "temperature_c" :(0,100),
    "pressure_bar" : (0,10),
    "speed_mps" : (0,20),
    "humidity_pct" : (0,100),
    "vibration_mm_s" : (0,20),
    "current_a" : (0,30)
    }

    df = load_senor_data(input_file) # 파일 읽어오기
    df = standardize_columns(df) # 컬럼명 표준화
    df = convert_time_and_units(df) # 시간 컬럼 및 단위 변환 
    df = clean_text_columns(df) # status 컬럼 정리 
    df = df.drop_duplicates().reset_index(drop=True) # 중복 데이터 제거 
    df = handle_missing_values(df) # 결측치 처리 
    df, outlier_report = remove_outliers(df, valid_ranges) # 이상치 제거, 리포트 생성 
    df = add_quality_result(df) # 품질 판정 컬럼 추가
    quality_report = make_quality_report(df) # 최종 품질 리포트 작성

    return df, outlier_report, quality_report

In [71]:
file_path = r"C:\Users\user\Desktop\python_campus\dev\data\raw\week07_day04_raw_messy_sensor.csv"

df, outlier_report, quality_report = run_processing(file_path)

In [72]:
df.isna().sum()

timestamp         0
machine_id        0
temperature_c     0
pressure_bar      0
speed_mps         0
humidity_pct      0
vibration_mm_s    0
current_a         0
status            0
pressure_kpa      0
speed_mpm         0
quality           0
dtype: int64

In [73]:
df.head()

,timestamp,machine_id,temperature_c,pressure_bar,speed_mps,humidity_pct,vibration_mm_s,current_a,status,pressure_kpa,speed_mpm,quality
0,2026-07-01 09:00:00,MC_03,28.36,3.48,5.28,51.35,3.89,10.39,UNKNOWN,348.0,316.8,OK
1,2026-07-01 09:01:00,MC_01,24.20,2.82,4.64,67.22,3.62,6.34,NG,282.0,278.4,OK
2,2026-07-01 09:02:00,MC_03,27.67,2.45,4.11,64.43,2.42,6.92,OK,245.0,246.6,OK
3,2026-07-01 09:03:00,MC_03,26.59,2.79,4.39,59.31,4.95,7.76,NG,279.0,263.4,OK
4,2026-07-01 09:04:00,MC_01,23.92,2.56,4.67,61.00,1.58,7.82,OK,256.0,280.2,OK


In [74]:
outlier_report

,column,min_value,max_value,outlier_count
0,temperature_c,0,100,1
1,pressure_bar,0,10,1
2,speed_mps,0,20,1
3,humidity_pct,0,100,1
4,vibration_mm_s,0,20,1
5,current_a,0,30,1


In [75]:
quality_report

,total_count,ok_count,ng_count,ng_ratio
0,113,113,0,0.0


temperatrue_c를 화씨 온도 temperature_f로 변환해 보세요.

In [79]:
mission_df = df.copy()

mission_df["temperature_f"] = mission_df["temperature_c"] * 9/5 + 32

mission_df[["temperature_c", "temperature_f"]]

,temperature_c,temperature_f
0,28.36,83.048
1,24.20,75.560
2,27.67,81.806
3,26.59,79.862
4,23.92,75.056
...,...,...
108,24.46,76.028
109,27.76,81.968
110,25.52,77.936
111,28.42,83.156


humidity_pct가 65 이상인 데이터만 필터링해서 개수를 확인해 보세요.

In [86]:
count = (mission_df["humidity_pct"]>=65).sum()

count

np.int64(15)

In [ ]:
# 모범답안

high_humidity_df = mission_df[mission_df["humidity_pct"]>=65]

print(len(high_humidity_df))

15


In [88]:
high_humidity_df[["timestamp", "machine_id", "humidity_pct"]].head(15)

,timestamp,machine_id,humidity_pct
1,2026-07-01 09:01:00,MC_01,67.22
7,2026-07-01 09:07:00,MC_02,66.67
16,2026-07-01 09:16:00,MC_02,65.61
23,2026-07-01 09:24:00,MC_01,70.28
33,2026-07-01 09:37:00,MC_03,66.02
39,2026-07-01 09:44:00,MC_01,71.87
47,2026-07-01 09:54:00,MC_02,67.62
59,2026-07-01 10:06:00,MC_02,65.28
63,2026-07-01 10:10:00,MC_03,71.87
64,2026-07-01 10:11:00,MC_02,65.07


설비별 평균 진동값과 평균 전류값을 계산해 보세요.

In [94]:
machine_avg = mission_df.groupby("machine_id").agg(
    avg_vib = ("vibration_mm_s", "mean"),
    avg_ele = ("current_a", "mean")
).reset_index()

machine_avg

,machine_id,avg_vib,avg_ele
0,MC_01,2.903611,8.890000
1,MC_02,3.092432,8.860811
2,MC_03,3.158000,8.802250


date 컬럼을 만들고, 날짜별 OK/NG 개수와 NG 비율 리포트를 만들어 저장해 보세요.

In [98]:
mission_df["date"] = mission_df["timestamp"].dt.date # 날짜만 뽑아줌 

ratio_report = mission_df.groupby("date").agg(
    total_count = ("quality", "count"),
    ok_count = ("status", lambda x: (x=="OK").sum()),
    ng_count = ("status", lambda x: (x=="NG").sum()),
).reset_index()

ratio_report["ng_ratio"] = (ratio_report["ng_count"] / ratio_report["total_count"] * 100).round(2)

report_path = "daily_report.csv"
ratio_report.to_csv(report_path, index=False, encoding="utf-8-sig")

ratio_report

,date,total_count,ok_count,ng_count,ng_ratio
0,2026-07-01,113,73,32,28.32
